In [42]:
from __future__ import annotations

import argparse
import math
from pathlib import Path
from typing import Dict, Optional
import pandas as pd
from lhotse import fix_manifests, validate_recordings_and_supervisions
from lhotse.audio import RecordingSet
from lhotse.kaldi import load_kaldi_data_dir
from lhotse.supervision import SupervisionSet
from lhotse.utils import Pathlike

In [43]:
EDACC_ROOT = Path("data/edacc")
OUTPUT_DIR = Path("data/edacc_filtered")

_EDACC_SAMPLING_RATE = 32000


In [44]:

def parse_linguistic_background(path: Pathlike) -> Dict[str, Dict]:
    df = pd.read_csv(path)
    df = df.rename(
        columns={
            "What is your gender?": "gender",
            "What’s your ethnic background? ": "ethnicity",
            "What is your higher level of education?": "education",
            "How would you describe your accent in English? (e.g. Italian, Glaswegian)": "accent",
            "Do you speak any second languages? separate them with commas  (e.g., Mandarin,Catalan,French )": "other_languages",
            "What’s your year of birth? (e.g., 1992)": "birth_year",
            "What year did you start learning English? (e.g., 1999)": "start_english_year",
        }
    )
    df["age"] = 2022 - df["birth_year"]
    df["years_speaking_english"] = 2022 - df["start_english_year"]

    def parse(key, val):
        if key == "years_speaking_english":
            if pd.isna(val):
                return None
            return int(val)
        if key == "other_languages":
            if isinstance(val, float) and math.isnan(val):
                return []
            if pd.isna(val):
                return []
            return [v.strip() for v in str(val).split(",") if v.strip()]
        if pd.isna(val):
            return None
        if isinstance(val, str):
            return val.strip()
        return val

    spk2meta = {
        str(row["PARTICIPANT_ID"]): {
            m: parse(m, row.get(m))
            for m in (
                "gender",
                "ethnicity",
                "education",
                "accent",
                "other_languages",
                "age",
                "years_speaking_english",
            )
        }
        for _, row in df.iterrows()
    }
    return spk2meta

def load_kaldi_map(path: Path, value_name: str, split_at_first_space: bool = False) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if split_at_first_space:
                utt_id, value = line.split(maxsplit=1)
            else:
                parts = line.split()
                if len(parts) < 2:
                    continue
                utt_id, value = parts[0], parts[1]
            rows.append((utt_id, value))
    return pd.DataFrame(rows, columns=["utt_id", value_name])


def load_segments(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 4:
                continue
            utt_id, recording_id, start, end = parts
            rows.append((utt_id, recording_id, float(start), float(end)))
    return pd.DataFrame(rows, columns=["utt_id", "recording_id", "start", "end"])


def find_jamaican_speakers(spk2meta: Dict[str, Dict], n_speakers: int = 4) -> list[str]:
    rows = []
    for spk, meta in spk2meta.items():
        blob = " ".join(str(v) for v in meta.values() if v is not None).lower()
        if "jamaic" in blob:
            rows.append(spk)
    rows = sorted(rows)
    return rows[:n_speakers]

In [45]:
def collect_split_dataframe(corpus_dir: Path, split: str, speaker_ids: set[str]) -> pd.DataFrame:
    utt2spk = load_kaldi_map(corpus_dir / split / "utt2spk", "speaker")
    text = load_kaldi_map(corpus_dir / split / "text", "text", split_at_first_space=True)
    segments = load_segments(corpus_dir / split / "segments")

    df = segments.merge(utt2spk, on="utt_id", how="inner").merge(text, on="utt_id", how="inner")
    df["speaker"] = df["speaker"].astype(str)
    df = df[df["speaker"].isin(speaker_ids)].copy()
    df["start"] = df["start"].astype(float)
    df["end"] = df["end"].astype(float)
    df["duration"] = df["end"] - df["start"]
    df["n_words"] = df["text"].astype(str).str.split().str.len()
    df = df[(df["duration"] <= 28.0) & (df["n_words"] >= 4)].copy()  # filter out very long segments and very short ones
    df["split"] = split
    df["l1"] = "Jamaican English"
    df["accent"] = "Jamaican English"
    df["corpus"] = "edacc"
    return df.sort_values(["speaker", "recording_id", "start", "end"]).reset_index(drop=True)




In [46]:
def cut_subset_audio(corpus_dir: Path, df: pd.DataFrame, out_dir: Path, split: str) -> pd.DataFrame:
    import soundfile as sf
    import tqdm
    
    wav_dir = out_dir / "wavs"
    wav_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for _, r in tqdm.tqdm(df.iterrows(), total=len(df)):
        src = corpus_dir / "data" / f"{r['recording_id']}.wav"
        if not src.exists():
            print(f"[WARN] missing source audio: {src}")
            continue

        audio, sr = sf.read(src)
        s = max(0, int(round(r["start"] * sr)))
        e = min(len(audio), int(round(r["end"] * sr)))
        if e <= s:
            continue
        chunk = audio[s:e]
        out_wav = wav_dir / f"{r['utt_id']}.wav"
        sf.write(out_wav, chunk, sr)

        rows.append(
            {
                "utteranceid": r["utt_id"],
                "speaker": r["speaker"],
                "l1": r["l1"],
                "accent": r["accent"],
                "text": r["text"],
                "wavpath": str(out_wav),
                "split": split,
                "corpus": r["corpus"],
                "recording_id": r["recording_id"],
                "start": r["start"],
                "end": r["end"],
                "duration": r["duration"],
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(out_dir / f"{split}_manifest.csv", index=False)
    return out



In [47]:
class Args:
    corpus_dir: str
    output_dir: str
    n_speakers: int
    max_utts: Optional[int]
    splits: str

    def __init__(self, corpus_dir: str, output_dir: str, n_speakers: int = 4, max_utts: Optional[int] = None, splits: str = "dev,test"):
        self.corpus_dir = corpus_dir
        self.output_dir = output_dir
        self.n_speakers = n_speakers
        self.max_utts = max_utts
        self.splits = splits

args = Args(
    corpus_dir="data/edacc",
    output_dir="data/edacc_jamaican_subset",
    n_speakers=4,
    max_utts=None,
    splits="dev,test",
)

corpus_dir = Path(args.corpus_dir) 
out_dir = Path(args.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

spk2meta = parse_linguistic_background(corpus_dir / "linguistic_background.csv")
jamaican_speakers = find_jamaican_speakers(spk2meta, n_speakers=args.n_speakers)
if not jamaican_speakers:
    raise RuntimeError("No Jamaican speakers found in linguistic_background.csv")
print("Selected Jamaican speakers:", jamaican_speakers)

all_rows = []
for split in [s.strip() for s in args.splits.split(",") if s.strip()]:
    df = collect_split_dataframe(corpus_dir, split, set(jamaican_speakers))
    if len(df) == 0:
        print(f"{split}: 0 utterances for selected speakers; falling back to all Jamaican speakers across corpus")
        for alt_split in ["dev", "test"]:
            alt_df = collect_split_dataframe(corpus_dir, alt_split, set(jamaican_speakers))
            if len(alt_df):
                df = alt_df
                split = alt_split
                print(f"Using {alt_split} instead, with {len(df)} utterances")
                break
    if args.max_utts is not None:
        df = df.head(args.max_utts).copy()
    print(f"{split}: {len(df)} utterances before cutting")
    out = cut_subset_audio(corpus_dir, df, out_dir, split)
    print(f"{split}: wrote {len(out)} utterances")
    if len(out):
        all_rows.append(out)

if all_rows:
    summary = pd.concat(all_rows, ignore_index=True)
    summary.sort_values("utteranceid", inplace=True)
    summary.to_csv(out_dir / "jamaican_subset_all.csv", index=False)
    print("Wrote:", out_dir / "jamaican_subset_all.csv")
else:
    print("No utterances found after all filters.")

Selected Jamaican speakers: ['EDACC-C48-A', 'EDACC-C48-B', 'EDACC-C49-A', 'EDACC-C49-B']
dev: 215 utterances before cutting


  0%|          | 0/215 [00:00<?, ?it/s]

100%|██████████| 215/215 [00:33<00:00,  6.49it/s]


dev: wrote 215 utterances
test: 54 utterances before cutting


100%|██████████| 54/54 [00:06<00:00,  7.92it/s]

test: wrote 54 utterances
Wrote: data/edacc_jamaican_subset/jamaican_subset_all.csv
